# Practice: build your own recommender (supermarket)
A tiny content-based recommender: **TF-IDF** vectors + **cosine similarity** over 10 supermarket items in 3 categories.

## 1. Catalogue
Ten items, three categories, one short description each.

In [1]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

catalogue = pd.DataFrame(
    [
        # Dairy
        ("Whole Milk",       "Dairy",   "fresh creamy whole milk for cereal, coffee and baking, rich in calcium"),
        ("Greek Yogurt",     "Dairy",   "thick creamy strained yogurt, high protein, great with honey, fruit and granola for breakfast"),
        ("Cheddar Cheese",   "Dairy",   "sharp aged cheddar cheese, perfect for sandwiches, grilling and melting on pasta"),
        # Bakery
        ("Sourdough Bread",  "Bakery",  "crusty artisan sourdough loaf with a tangy flavour, ideal for toast, sandwiches and cheese"),
        ("Butter Croissant", "Bakery",  "flaky buttery French pastry, baked fresh each morning, great with coffee for breakfast"),
        ("Blueberry Muffin", "Bakery",  "soft sweet muffin baked with juicy blueberries, a quick breakfast or snack with coffee"),
        # Produce
        ("Bananas",          "Produce", "sweet ripe yellow bananas, a healthy snack, great in smoothies, cereal and baking"),
        ("Blueberries",      "Produce", "fresh juicy blueberries, sweet antioxidant fruit for yogurt, granola, smoothies and muffins"),
        ("Baby Spinach",     "Produce", "tender fresh green spinach leaves for salads, smoothies and cooking pasta"),
        ("Cherry Tomatoes",  "Produce", "sweet bite-sized tomatoes, perfect for salads, sandwiches and pasta sauce"),
    ],
    columns=["item", "category", "description"],
)
catalogue

,item,category,description
0,Whole Milk,Dairy,"fresh creamy whole milk for cereal, coffee and..."
1,Greek Yogurt,Dairy,"thick creamy strained yogurt, high protein, gr..."
2,Cheddar Cheese,Dairy,"sharp aged cheddar cheese, perfect for sandwic..."
3,Sourdough Bread,Bakery,crusty artisan sourdough loaf with a tangy fla...
4,Butter Croissant,Bakery,"flaky buttery French pastry, baked fresh each ..."
5,Blueberry Muffin,Bakery,soft sweet muffin baked with juicy blueberries...
6,Bananas,Produce,"sweet ripe yellow bananas, a healthy snack, gr..."
7,Blueberries,Produce,"fresh juicy blueberries, sweet antioxidant fru..."
8,Baby Spinach,Produce,"tender fresh green spinach leaves for salads, ..."
9,Cherry Tomatoes,Produce,"sweet bite-sized tomatoes, perfect for salads,..."


## 2. Vectorise the descriptions
A small tokenizer lowercases, drops stop words and strips plural "s" (so *smoothie* matches *smoothies*).

In [2]:
def tokenize(text):
    words = re.findall(r"[a-z]+", text.lower())
    return [w[:-1] if len(w) > 3 and w.endswith("s") and not w.endswith("ss") else w
            for w in words if w not in ENGLISH_STOP_WORDS]

vectorizer = TfidfVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None, sublinear_tf=True)
item_vectors = vectorizer.fit_transform(catalogue["description"])
print(item_vectors.shape, "= items x vocabulary terms")

(10, 63) = items x vocabulary terms


## 3. `recommend()`

In [4]:
def recommend(query, top_n=3, category=None):
    """Top-n catalogue items most similar to free-text `query`."""
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, item_vectors).ravel()
    result = catalogue.assign(score=scores)
    if category:
        result = result[result["category"] == category]
    result = result[result["score"] > 0].sort_values("score", ascending=False)
    return result.head(top_n)[["item", "category", "score"]].reset_index(drop=True)

def recommend_similar_to(item_name, top_n=3):
    """'Customers who like X may also like...'"""
    idx = catalogue.index[catalogue["item"] == item_name][0]
    scores = cosine_similarity(item_vectors[idx], item_vectors).ravel()
    scores[idx] = -1
    top = scores.argsort()[::-1][:top_n]
    return catalogue.iloc[top].assign(score=scores[top])[["item", "category", "score"]].reset_index(drop=True)

## 4. Try it

In [5]:
for q in ["something sweet and fruity for breakfast with coffee",
          "healthy ingredients for a salad",
          "I need things for a cheese sandwich",
          "smoothie ingredients"]:
    print(f"\nQuery: {q!r}")
    display(recommend(q))


Query: 'something sweet and fruity for breakfast with coffee'


,item,category,score
0,Blueberry Muffin,Bakery,0.464401
1,Butter Croissant,Bakery,0.318438
2,Whole Milk,Dairy,0.179410



Query: 'healthy ingredients for a salad'


,item,category,score
0,Bananas,Produce,0.274367
1,Cherry Tomatoes,Produce,0.208272
2,Baby Spinach,Produce,0.204258



Query: 'I need things for a cheese sandwich'


,item,category,score
0,Cheddar Cheese,Dairy,0.309348
1,Sourdough Bread,Bakery,0.279119



Query: 'smoothie ingredients'


,item,category,score
0,Blueberries,Produce,0.285976
1,Baby Spinach,Produce,0.275907
2,Bananas,Produce,0.267822


In [6]:
recommend_similar_to("Greek Yogurt")

,item,category,score
0,Blueberries,Produce,0.294785
1,Butter Croissant,Bakery,0.139899
2,Whole Milk,Dairy,0.102977


In [7]:
recommend("sweet snack", category="Produce")

,item,category,score
0,Bananas,Produce,0.387827
1,Blueberries,Produce,0.156103
2,Cherry Tomatoes,Produce,0.153566
